[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/41.%20The%20Terminal%20Concession%20Bidding%20Problem/P41-Tier-1_executed.ipynb)

---

# 41. The Terminal Concession Bidding Problem

## Tier 1: Integer Programming Formulation

### Goal and outputs

This notebook formulates a terminal concession award as a mixed-integer optimization problem. It first calculates weighted bidder scores, then uses binary decision variables to select exactly one eligible bidder. The notebook reports the solver status, selected bidder, score, and the value assigned to a payment variable, and examines how the selected bidder changes as the minimum score threshold varies.

The model has one objective: maximize the selected bidder's weighted score. It is not a multi-objective model. Financial capacity limits a payment variable, but payment is not part of the objective; consequently, the model does not determine an economically meaningful payment amount.

### Inputs and assumptions

The example contains three bidders (MTC, GPH, and ACS), four weighted evaluation criteria, a minimum score threshold, a concession duration, and an annual throughput target. Each criterion has a minimum score that bidders must meet. For this teaching example, the numerical criterion inputs are treated as score points in the weighted sum. Financial capacity is also used separately as an upper bound, measured in million euros, on the payment variable. This scoring convention is a simplifying modeling assumption, not a conversion between money and score points.

The model assumes that exactly one bidder must win and that bidders failing any criterion minimum or the overall score threshold cannot be selected. It does not model bidding strategies, uncertainty, the timing of payments, revenue over the concession period, employment commitments, automation investments, or throughput constraints. Concession duration and throughput target are recorded as problem data but do not enter the optimization formulation.

### Solution sequence

1. Define bidders, criteria, and problem parameters.
2. Calculate each bidder's weighted score and eligibility.
3. Formulate and solve the mixed-integer linear program using PuLP and CBC.
4. Visualize the scores, capacities, and criterion weights.
5. Repeat the same optimization model across a range of qualification thresholds.

A mixed-integer linear program (MILP) combines integer decisions, such as whether a bidder is selected, with continuous quantities, such as a payment amount. For this finite example, the solver searches for a globally optimal solution to the stated model. That guarantee applies to the mathematical model, so the interpretation of the result depends on the assumptions and objective described above.

In [ ]:
# These libraries support the data tables, plots, and mixed-integer model.
import math
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pulp import (
    LpMaximize,
    LpProblem,
    LpStatus,
    LpStatusOptimal,
    LpVariable,
    PULP_CBC_CMD,
    lpSum,
)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Bidder:
    """Store a bidder's evaluation inputs and commitments."""

    name: str
    financial_capacity: float  # Maximum payment capacity, in million euros
    technical_experience: int  # Evaluation input on the 0-100 scale
    revenue_commitment: float  # Annual commitment, in million euros
    performance_guarantee: int  # Evaluation input on the 0-100 scale
    employment_commitment: int  # Number of direct jobs
    automation_investment: float  # Investment, in million euros


@dataclass
class Criterion:
    """Store an evaluation criterion, its weight, and its minimum score."""

    name: str
    weight: float
    min_score: int = 0


@dataclass
class ConcessionProblem:
    """Collect the data and qualification parameters for the model."""

    bidders: List[Bidder]
    criteria: List[Criterion]
    min_qualification_threshold: float
    concession_duration: int  # Years
    annual_throughput_target: float  # Million TEU

In [ ]:
def create_concession_example() -> ConcessionProblem:
    """Create the three-bidder terminal concession example."""
    bidders = [
        Bidder("MTC", 85, 90, 80, 95, 280, 45),
        Bidder("GPH", 95, 85, 90, 85, 320, 65),
        Bidder("ACS", 80, 95, 85, 90, 295, 55),
    ]

    criteria = [
        Criterion("Financial Capacity", 0.30, 70),
        Criterion("Technical Experience", 0.25, 75),
        Criterion("Revenue Commitment", 0.25, 70),
        Criterion("Performance Guarantees", 0.20, 80),
    ]

    return ConcessionProblem(
        bidders=bidders,
        criteria=criteria,
        min_qualification_threshold=85.0,
        concession_duration=25,
        annual_throughput_target=1.5,
    )


concession_problem = create_concession_example()
print(
    f"Created concession problem with {len(concession_problem.bidders)} "
    f"bidders and {len(concession_problem.criteria)} criteria."
)

## Weighted evaluation and eligibility

A weighted score combines several evaluation criteria into one number. If bidder \(i\) receives score \(s_{ik}\) on criterion \(k\), and that criterion has weight \(w_k\), the total score is

$$
S_i = \sum_{k \in K} w_k s_{ik},
\qquad \sum_{k \in K} w_k = 1.
$$

Here, \(K\) is the set of criteria, \(w_k\) is the importance assigned to criterion \(k\), and \(S_i\) is bidder \(i\)'s weighted score. A bidder is eligible only if it meets every criterion-specific minimum and its total score is at least the problem's minimum qualification threshold. These two tests are both applied: meeting the total threshold does not compensate for failing an individual minimum.

In [ ]:
def calculate_weighted_scores(problem: ConcessionProblem) -> pd.DataFrame:
    """Calculate weighted scores and full eligibility for every bidder."""
    total_weight = sum(criterion.weight for criterion in problem.criteria)
    if not math.isclose(total_weight, 1.0, rel_tol=0.0, abs_tol=1e-9):
        raise ValueError("Criterion weights must sum to 1.0.")

    scores_data = []

    for bidder in problem.bidders:
        criterion_scores = {
            "Financial Capacity": bidder.financial_capacity,
            "Technical Experience": bidder.technical_experience,
            "Revenue Commitment": bidder.revenue_commitment,
            "Performance Guarantees": bidder.performance_guarantee,
        }

        total_score = sum(
            criterion_scores[criterion.name] * criterion.weight
            for criterion in problem.criteria
        )
        meets_criterion_minimums = all(
            criterion_scores[criterion.name] >= criterion.min_score
            for criterion in problem.criteria
        )
        qualified = (
            meets_criterion_minimums
            and total_score >= problem.min_qualification_threshold
        )

        scores_data.append(
            {
                "Bidder": bidder.name,
                "Financial Capacity": criterion_scores["Financial Capacity"],
                "Technical Experience": criterion_scores["Technical Experience"],
                "Revenue Commitment": criterion_scores["Revenue Commitment"],
                "Performance Guarantees": criterion_scores[
                    "Performance Guarantees"
                ],
                "Total Score": total_score,
                "Meets Criterion Minimums": meets_criterion_minimums,
                "Qualified": qualified,
            }
        )

    return pd.DataFrame(scores_data)


scores_df = calculate_weighted_scores(concession_problem)
print("Terminal Concession Bidding Evaluation")
print(scores_df.to_string(index=False))
print(
    "\nMinimum qualification threshold: "
    f"{concession_problem.min_qualification_threshold:.1f}"
)

## Mixed-integer optimization model

The optimization step chooses one eligible bidder with the highest weighted score. Let \(I\) be the set of bidders, \(K\) the set of criteria, \(x_i\) a binary selection decision, and \(z_i\) the payment variable for bidder \(i\). Let \(S_i\) be the weighted score, \(F_i\) the financial capacity, and \(\tau\) the minimum total-score threshold. Let \(Q_i=1\) when bidder \(i\) meets every criterion-specific minimum, and \(Q_i=0\) otherwise.

The model is

$$
\begin{aligned}
\max\quad & \sum_{i\in I} S_i x_i \\
\text{subject to}\quad
    & \sum_{i\in I} x_i = 1, \\
    & x_i = 0 && \text{for each } i\in I \text{ with } Q_i=0, \\
    & S_i x_i \geq \tau x_i && \text{for each } i\in I, \\
    & 0 \leq z_i \leq F_i x_i && \text{for each } i\in I, \\
    & x_i\in\{0,1\},\quad z_i\geq 0 && \text{for each } i\in I.
\end{aligned}
$$

The first constraint requires exactly one winner. The second excludes bidders who fail an individual criterion minimum. The third applies the overall score threshold to a selected bidder; for an unselected bidder, both sides are zero. The last bound makes payment zero for unselected bidders and limits a selected bidder's payment by its capacity. The objective maximizes the winner's score because exactly one \(x_i\) equals one.

In the code, `x` represents the binary variables and `z` the continuous payment variables. The score and eligibility data come from `calculate_weighted_scores`, and the constraints are added to the PuLP model in `solve_concession_bidding_mip`. No objective term rewards a larger `z_i`, and there is no lower payment requirement. Therefore, the selected bidder and optimal score are determined, but the payment value is not: any feasible payment from zero to the winner's capacity has the same objective value. A solver-reported payment value must not be interpreted as an optimized concession payment.

With \(|I|\) bidders, the formulation has \(|I|\) binary and \(|I|\) continuous variables. The optimization problem is a single-winner selection model; its integer decision structure is handled by the MILP solver. If no bidder is eligible, the single-winner and eligibility constraints make the model infeasible.

In [ ]:
def solve_concession_bidding_mip(
    problem: ConcessionProblem,
) -> Tuple[Dict[str, object], LpProblem]:
    """Solve the single-winner mixed-integer concession model."""
    model = LpProblem("Terminal_Concession_Bidding", LpMaximize)

    # Binary selection records the winner; continuous payment records a
    # feasible amount bounded by capacity, but is not rewarded by the objective.
    x = {
        bidder.name: LpVariable(f"select_{bidder.name}", cat="Binary")
        for bidder in problem.bidders
    }
    z = {
        bidder.name: LpVariable(
            f"payment_{bidder.name}", lowBound=0, cat="Continuous"
        )
        for bidder in problem.bidders
    }

    scores_df = calculate_weighted_scores(problem)
    score_by_bidder = scores_df.set_index("Bidder")["Total Score"].to_dict()
    qualified_by_bidder = (
        scores_df.set_index("Bidder")["Qualified"].to_dict()
    )

    model += lpSum(
        score_by_bidder[bidder.name] * x[bidder.name]
        for bidder in problem.bidders
    )
    model += lpSum(x[bidder.name] for bidder in problem.bidders) == 1, "Single_Winner"

    for bidder in problem.bidders:
        name = bidder.name
        if not qualified_by_bidder[name]:
            model += x[name] == 0, f"Disqualified_{name}"

        model += (
            z[name] <= bidder.financial_capacity * x[name],
            f"Financial_Capacity_{name}",
        )
        model += (
            score_by_bidder[name] * x[name]
            >= problem.min_qualification_threshold * x[name],
            f"Qualification_Threshold_{name}",
        )

    print("Solving the mixed-integer programming formulation...")
    model.solve(PULP_CBC_CMD(msg=False))

    results: Dict[str, object] = {
        "status": LpStatus[model.status],
        "winner": None,
        "winning_score": 0.0,
        "payment_commitment": 0.0,
        "all_scores": scores_df,
    }

    if model.status == LpStatusOptimal:
        selected_bidders = [
            bidder
            for bidder in problem.bidders
            if x[bidder.name].value() is not None
            and x[bidder.name].value() > 0.5
        ]
        if len(selected_bidders) != 1:
            raise RuntimeError(
                "An optimal solution must select exactly one bidder."
            )

        winner = selected_bidders[0]
        results["winner"] = winner.name
        results["winning_score"] = score_by_bidder[winner.name]
        results["payment_commitment"] = z[winner.name].value()

    return results, model


results, model = solve_concession_bidding_mip(concession_problem)
print(f"\nOptimization status: {results['status']}")
if results["winner"] is not None:
    print(f"Selected bidder: {results['winner']}")
    print(f"Winning score: {results['winning_score']:.2f}")
    print(
        "Solver-selected payment variable value (not optimized): "
        f"€{results['payment_commitment']:.1f} million"
    )

## Visualizing the evaluation

The charts summarize the input values, weighted scores, financial capacities, and criterion weights. The capability comparison uses the numerical inputs as score points under the example's scoring convention; it does not imply that monetary amounts and experience or guarantee ratings have the same physical units. The capacity chart displays financial capacity separately from the solver-selected payment-variable value, which the objective does not determine.

In [ ]:
def visualize_concession_results(
    problem: ConcessionProblem, results: Dict[str, object]
):
    """Plot bidder inputs, weighted scores, capacities, and criterion weights."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle("Terminal Concession Bidding Analysis", fontsize=16)

    categories = [criterion.name for criterion in problem.criteria]
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    closed_angles = angles + angles[:1]
    ax1 = axes[0, 0]

    for bidder in problem.bidders:
        values = [
            bidder.financial_capacity,
            bidder.technical_experience,
            bidder.revenue_commitment,
            bidder.performance_guarantee,
        ]
        closed_values = values + values[:1]
        ax1.plot(closed_angles, closed_values, marker="o", label=bidder.name)
        ax1.fill(closed_angles, closed_values, alpha=0.15)

    ax1.set_xticks(angles)
    ax1.set_xticklabels(categories)
    ax1.set_ylim(0, 100)
    ax1.set_title("Recorded Criterion Inputs (Score-Point Convention)")
    ax1.set_ylabel("Numerical input (score points)")
    ax1.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
    ax1.grid(True)

    scores_df = results["all_scores"]
    ax2 = axes[0, 1]
    x_positions = np.arange(len(scores_df))
    bars = ax2.bar(x_positions, scores_df["Total Score"], width=0.6)
    ax2.axhline(
        y=problem.min_qualification_threshold,
        color="red",
        linestyle="--",
        label=f"Minimum threshold ({problem.min_qualification_threshold:g})",
    )

    if results["winner"] is not None:
        winner_index = scores_df.index[
            scores_df["Bidder"] == results["winner"]
        ][0]
        bars[winner_index].set_edgecolor("gold")
        bars[winner_index].set_linewidth(3)

    ax2.set_xlabel("Bidder")
    ax2.set_ylabel("Weighted score (points)")
    ax2.set_title("Weighted Evaluation Scores")
    ax2.set_xticks(x_positions)
    ax2.set_xticklabels(scores_df["Bidder"])
    ax2.legend()
    ax2.grid(axis="y", alpha=0.3)

    ax3 = axes[1, 0]
    names = [bidder.name for bidder in problem.bidders]
    capacities = [bidder.financial_capacity for bidder in problem.bidders]
    capacity_colors = [
        "#FF6B6B" if name == results["winner"] else "#4ECDC4"
        for name in names
    ]
    ax3.bar(names, capacities, color=capacity_colors, alpha=0.75, label="Capacity")

    payment_value = results["payment_commitment"]
    if results["winner"] is not None and payment_value is not None:
        winner_index = names.index(results["winner"])
        ax3.scatter(
            [names[winner_index]],
            [payment_value],
            color="gold",
            edgecolor="black",
            zorder=3,
            label="Solver-selected payment variable",
        )

    ax3.set_xlabel("Bidder")
    ax3.set_ylabel("Financial capacity (million euros)")
    ax3.set_title("Financial Capacity and Payment Variable")
    ax3.legend()
    ax3.grid(axis="y", alpha=0.3)

    ax4 = axes[1, 1]
    criterion_names = [criterion.name for criterion in problem.criteria]
    weights = [criterion.weight for criterion in problem.criteria]
    ax4.pie(
        weights,
        labels=criterion_names,
        autopct="%1.1f%%",
        startangle=90,
    )
    ax4.set_title("Evaluation Criterion Weights")

    plt.tight_layout()
    plt.show()
    return fig


fig = visualize_concession_results(concession_problem, results)

## Sensitivity analysis: qualification threshold

Sensitivity analysis asks how the model's result responds when one input changes while the other data and the formulation remain fixed. Here, the minimum total-score threshold is varied from 70 to 95 in steps of 2.5. For each threshold, the same MILP is solved again, and the output records the selected bidder, its score, and the number of bidders who meet both the criterion-specific minimums and the current total-score threshold.

A higher threshold can remove bidders from consideration. If no bidder remains eligible, the model cannot satisfy its exactly-one-winner constraint and has no optimal winner. The original threshold is restored after the analysis, including if an error occurs, so later notebook cells continue to use the main example's threshold.

In [ ]:
def sensitivity_analysis(problem: ConcessionProblem) -> pd.DataFrame:
    """Solve the model over a range of minimum score thresholds."""
    thresholds = np.arange(70, 96, 2.5)
    sensitivity_results = []
    original_threshold = problem.min_qualification_threshold

    try:
        for threshold in thresholds:
            problem.min_qualification_threshold = float(threshold)
            results, _ = solve_concession_bidding_mip(problem)
            qualified_count = int(results["all_scores"]["Qualified"].sum())

            sensitivity_results.append(
                {
                    "Threshold": float(threshold),
                    "Winner": results["winner"] or "No Solution",
                    "Winning Score": results["winning_score"],
                    "Qualified Bidders": qualified_count,
                    "Status": results["status"],
                }
            )
    finally:
        problem.min_qualification_threshold = original_threshold

    return pd.DataFrame(sensitivity_results)


sensitivity_df = sensitivity_analysis(concession_problem)
print("Sensitivity Analysis on Qualification Threshold")
print(sensitivity_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
thresholds = sensitivity_df["Threshold"]
winners = sensitivity_df["Winner"]
unique_winners = winners.unique()
colors = plt.cm.Set2(np.linspace(0, 1, len(unique_winners)))
winner_colors = dict(zip(unique_winners, colors))

for winner in unique_winners:
    mask = winners == winner
    axes[0].plot(
        thresholds[mask],
        np.ones(int(mask.sum())),
        marker="o",
        linestyle="-",
        color=winner_colors[winner],
        label=winner,
    )

axes[0].set_xlabel("Minimum qualification threshold (score points)")
axes[0].set_ylabel("Selected outcome")
axes[0].set_title("Selected Bidder by Qualification Threshold")
axes[0].set_yticks([1])
axes[0].set_yticklabels(["Outcome recorded"])
axes[0].legend(title="Winner")
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    thresholds,
    sensitivity_df["Qualified Bidders"],
    marker="s",
    color="#E74C3C",
)
axes[1].set_xlabel("Minimum qualification threshold (score points)")
axes[1].set_ylabel("Number of qualified bidders")
axes[1].set_title("Eligible Bidders by Qualification Threshold")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary report

The report gathers the problem configuration, optimization status, selected bidder, and sensitivity results. It also reports the actual number of variables and constraints in the constructed PuLP model rather than inferring model size from the bidder count. The payment variable is identified as a feasible solver value, not as a payment optimized by this objective.

In [ ]:
def generate_summary_report(
    problem: ConcessionProblem,
    results: Dict[str, object],
    sensitivity_df: pd.DataFrame,
    model: LpProblem,
) -> None:
    """Print the model configuration, solution, and sensitivity summary."""
    print("\n" + "=" * 80)
    print("TERMINAL CONCESSION BIDDING - INTEGER PROGRAMMING ANALYSIS")
    print("=" * 80)

    print("\nProblem configuration:")
    print(f"- Number of bidders: {len(problem.bidders)}")
    print(f"- Number of criteria: {len(problem.criteria)}")
    print(f"- Concession duration: {problem.concession_duration} years")
    print(
        "- Annual throughput target: "
        f"{problem.annual_throughput_target} million TEU"
    )
    print(
        "- Minimum qualification threshold: "
        f"{problem.min_qualification_threshold:g} score points"
    )

    print("\nOptimization results:")
    print(f"- Status: {results['status']}")
    if results["winner"] is not None:
        print(f"- Selected bidder: {results['winner']}")
        print(f"- Winning score: {results['winning_score']:.2f}")
        print(
            "- Solver-selected payment variable value (not optimized): "
            f"€{results['payment_commitment']:.1f} million"
        )

        winner = next(
            bidder
            for bidder in problem.bidders
            if bidder.name == results["winner"]
        )
        print("\nSelected bidder profile:")
        print(f"- Financial capacity: €{winner.financial_capacity} million")
        print(f"- Technical experience: {winner.technical_experience}/100")
        print(
            f"- Revenue commitment: €{winner.revenue_commitment} million/year"
        )
        print(
            f"- Performance guarantee: {winner.performance_guarantee}/100"
        )
        print(f"- Employment commitment: {winner.employment_commitment} jobs")
        print(
            f"- Automation investment: €{winner.automation_investment} million"
        )

    print("\nBidder evaluation:")
    scores_df = results["all_scores"]
    for _, row in scores_df.iterrows():
        eligibility = "Qualified" if row["Qualified"] else "Not qualified"
        selected = " (selected)" if row["Bidder"] == results["winner"] else ""
        print(
            f"- {row['Bidder']}: {row['Total Score']:.2f} "
            f"({eligibility}){selected}"
        )

    print("\nSensitivity analysis:")
    outcome_changes = []
    for index in range(1, len(sensitivity_df)):
        if (
            sensitivity_df.iloc[index]["Winner"]
            != sensitivity_df.iloc[index - 1]["Winner"]
        ):
            outcome_changes.append(sensitivity_df.iloc[index]["Threshold"])

    if outcome_changes:
        print(
            "- Outcome changes at tested thresholds: "
            f"{outcome_changes}"
        )
    else:
        print("- The selected outcome is stable across the tested thresholds.")

    feasible_winners = sensitivity_df.loc[
        sensitivity_df["Winner"] != "No Solution", "Winner"
    ]
    if not feasible_winners.empty:
        most_frequent_winner = feasible_winners.mode().iloc[0]
        print(
            "- Most frequent winner among feasible sensitivity cases: "
            f"{most_frequent_winner}"
        )

    print(
        "- Qualified-bidder count range: "
        f"{sensitivity_df['Qualified Bidders'].min()} to "
        f"{sensitivity_df['Qualified Bidders'].max()}"
    )

    binary_variable_count = sum(
        variable.cat == "Integer" and variable.lowBound == 0 and variable.upBound == 1
        for variable in model.variables()
    )
    print("\nModel and interpretation:")
    print(f"- Total variables: {len(model.variables())}")
    print(f"- Binary selection variables: {binary_variable_count}")
    print(f"- Constraints: {len(model.constraints)}")
    print(
        "- The objective maximizes weighted score; the payment variable is "
        "capacity-bounded but not optimized."
    )
    print(
        "- Results depend on the stated scoring convention and model scope; "
        "duration, throughput, employment, and automation do not constrain "
        "the optimization."
    )
    print("=" * 80)


generate_summary_report(concession_problem, results, sensitivity_df, model)